In [1]:
import os
import unicodedata

import torch
import pandas as pd
from tqdm import tqdm
import fitz  # PyMuPDF
import concurrent.futures
from functools import partial
import logging

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)
from accelerate import Accelerator

# Langchain 관련
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from vllm import LLM, SamplingParams


In [2]:
def process_pdf_improved(file_path, chunk_size=800, chunk_overlap=50):
    """PDF 텍스트 추출 후 chunk 단위로 나누기 (개선된 버전)"""
    doc = fitz.open(file_path)
    chunks = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    for page_num, page in enumerate(doc):
        text = page.get_text()
        page_chunks = splitter.split_text(text)
        
        for chunk in page_chunks:
            chunks.append(Document(
                page_content=chunk,
                metadata={
                    "source": file_path,
                    "page": page_num + 1,  # 1-based page numbering
                    "total_pages": len(doc),
                    "chunk_size": chunk_size,
                    "chunk_overlap": chunk_overlap
                }
            ))
    
    return chunks

In [3]:
def create_vector_db_improved(chunks, model_path="BAAI/bge-m3", use_gpu=True, batch_size=64):
    # GPU 사용 가능 여부 확인
    if use_gpu and not torch.cuda.is_available():
        print("GPU is not available. Falling back to CPU.")
        use_gpu = False
    
    device = "cuda" if use_gpu else "cpu"
    print(f"Using device: {device}")

    # 임베딩 모델 설정
    model_kwargs = {'device': device}
    encode_kwargs = {
        'normalize_embeddings': True,
        'batch_size': batch_size,
        'device': device
    }
    
    try:
        # torch.set_grad_enabled(False)를 사용하여 그래디언트 계산 비활성화
        with torch.no_grad():
            embeddings = HuggingFaceEmbeddings(
                model_name=model_path,
                model_kwargs=model_kwargs,
                encode_kwargs=encode_kwargs
            )
            
            # FAISS DB 생성 (GPU 인덱스 사용)
            if use_gpu:
                index = faiss.IndexFlatIP(embeddings.embedding_dim)
                index = faiss.index_cpu_to_gpu(faiss.StandardGpuResources(), 0, index)
                db = FAISS(embeddings.embed_query, index)
                db.add_documents(chunks)
            else:
                db = FAISS.from_documents(chunks, embedding=embeddings)
            
            # 메타데이터 저장
            db.metadata = {
                "model_path": model_path,
                "device": device,
                "batch_size": batch_size,
                "num_documents": len(chunks),
                "gpu_used": use_gpu
            }
            
            return db
    
    except Exception as e:
        print(f"Error creating vector DB: {str(e)}")
        return None

In [4]:
def normalize_path(path):
    """경로 유니코드 정규화"""
    return unicodedata.normalize('NFC', path)

In [5]:
def process_pdfs_from_dataframe(df, base_directory, max_workers=4, chunk_size=800, chunk_overlap=50, use_gpu=True):
    """데이터프레임에서 PDF 처리 및 벡터 DB, retriever 생성 (개선된 버전)"""
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    logger = logging.getLogger(__name__)
    
    pdf_databases = {}
    unique_paths = df['Source_path'].unique()
    
    def process_single_pdf(path, base_dir, chunk_size, chunk_overlap, use_gpu):
        try:
            normalized_path = normalize_path(path)
            full_path = os.path.normpath(os.path.join(base_dir, normalized_path.lstrip('./'))) if not os.path.isabs(normalized_path) else normalized_path
            
            pdf_title = os.path.splitext(os.path.basename(full_path))[0]
            logger.info(f"Processing {pdf_title}...")
            
            chunks = process_pdf_improved(full_path, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
            db = create_vector_db_improved(chunks, use_gpu=use_gpu)
            
            retriever = db.as_retriever(search_type="mmr", search_kwargs={'k': 3, 'fetch_k': 8})
            
            return pdf_title, {'db': db, 'retriever': retriever}
        except Exception as e:
            logger.error(f"Error processing {path}: {str(e)}")
            return None
    
    process_func = partial(process_single_pdf, base_dir=base_directory, 
                           chunk_size=chunk_size, chunk_overlap=chunk_overlap, use_gpu=use_gpu)
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_path = {executor.submit(process_func, path): path for path in unique_paths}
        
        for future in tqdm(concurrent.futures.as_completed(future_to_path), total=len(unique_paths), desc="Processing PDFs"):
            path = future_to_path[future]
            try:
                result = future.result()
                if result:
                    pdf_title, data = result
                    pdf_databases[pdf_title] = data
            except Exception as e:
                logger.error(f"Unexpected error processing {path}: {str(e)}")
    
    logger.info(f"Processed {len(pdf_databases)} PDFs successfully.")
    return pdf_databases

In [6]:
base_directory = './' # Your Base Directory
df = pd.read_csv('./test.csv')
pdf_databases = process_pdfs_from_dataframe(df, base_directory)

2024-08-01 17:00:34,234 - INFO - Processing 중소벤처기업부_혁신창업사업화자금(융자)...
2024-08-01 17:00:34,235 - INFO - Processing 보건복지부_부모급여(영아수당) 지원...
2024-08-01 17:00:34,245 - INFO - Processing 보건복지부_노인장기요양보험 사업운영...
2024-08-01 17:00:34,447 - INFO - Processing 산업통상자원부_에너지바우처...


Using device: cuda
Using device: cuda
Using device: cuda


Processing PDFs:   0%|          | 0/9 [00:00<?, ?it/s]

Using device: cuda


/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
2024-08-01 17:00:35,012 - INFO - PyTorch version 2.3.1 available.
2024-08-01 17:00:35,126 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
2024-08-01 17:00:35,127 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
2024-08-01 17:00:35,127 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
2024-08-01 17:00:35,128 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
2024-08-01 17:00:41,763 - INFO - Loading faiss with AVX2 support.
2024-08-01 17:00:41,764 - INFO - Could not load library with

...

Processing PDFs:  11%|█         | 1/9 [00:12<01:38, 12.30s/it]
2024-08-01 17:00:47,880 - INFO - Processing 「FIS 이슈 & 포커스」 22-4호 《중앙-지방 간 재정조정제도》...
2024-08-01 17:00:47,992 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


Using device: cuda


2024-08-01 17:00:57,283 - INFO - Processing 「FIS 이슈 & 포커스」 23-2호 《핵심재정사업 성과관리》...
2024-08-01 17:00:57,379 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


Using device: cuda


2024-08-01 17:00:58,432 - INFO - Processing 「FIS 이슈&포커스」 22-2호 《재정성과관리제도》...


In [6]:
def setup_llm_pipeline():


    # 모델 ID 
    model_id = "rtzr/ko-gemma-2-9b-it"

    # 토크나이저 로드 및 설정
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.use_default_system_prompt = False



    # HuggingFacePipeline 객체 생성
    text_generation_pipeline = pipeline(
        model=model_id,
        tokenizer=tokenizer,
        task="text-generation",
        temperature=0.2,
        return_full_text=False,
        max_new_tokens=512,
        device_map="auto"
    )

    llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

    return llm

In [7]:
# LLM 파이프라인
llm = setup_llm_pipeline()

Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [ ]:
# from langchain_community.llms import VLLM
# import multiprocessing

# # 멀티프로세싱 시작 방법을 'spawn'으로 설정
# multiprocessing.set_start_method('spawn', force=True)


# llm = VLLM(
#     model="rtzr/ko-gemma-2-9b-it",
#     tensor_parallel_size=8,
#     temperature=0,
#     top_k=50,
#     top_p=0.95,
#     max_model_len=1024,
#     vllm_kwargs={'gpu_memory_utilization': 0.6}
# )

# print(llm("model test"))

In [ ]:
def normalize_string(s):
    """유니코드 정규화"""
    return unicodedata.normalize('NFC', s)

def format_docs(docs):
    """검색된 문서들을 하나의 문자열로 포맷팅"""
    context = ""
    for doc in docs:
        context += doc.page_content
        context += '\n'
    return context

In [ ]:
# 결과를 저장할 리스트 초기화
results = []

# DataFrame의 각 행에 대해 처리
for _, row in tqdm(df.iterrows(), total=len(df), desc="Answering Questions"):
    # 소스 문자열 정규화
    source = normalize_string(row['Source'])
    question = row['Question']

    # 정규화된 키로 데이터베이스 검색
    normalized_keys = {normalize_string(k): v for k, v in pdf_databases.items()}
    retriever = normalized_keys[source]['retriever']

    # RAG 체인 구성
    template = """
    당신은 재정정보 검색을 돕는 AI입니다. 아래의 정보를 바탕으로 질문에 맞는 정확하고 자세한 답변을 제공하세요.
    가능하다면 추가적인 맥락이나 관련 정보를 포함하여 사용자의 이해를 돕도록 하세요. 다음 지침을 따르세요:

    1. 제공된 정보 내에서만 답변하세요. 추가적인 정보가 필요하다면 "정보가 부족합니다"라고 답변하세요.
    2. 질문의 맥락에 맞추어 관련 있는 정보를 최대한 포함하세요.
    3. 답변이 명확하고 이해하기 쉽게 작성하세요.
    4. 질문이 모호할 경우, 가능한 해석을 바탕으로 답변을 제공하세요.
    
    {context}

    질문: {question}

    답변:
    """
    prompt = PromptTemplate.from_template(template)

    # RAG 체인 정의
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # 답변 추론
    print(f"Question: {question}")
    full_response = rag_chain.invoke(question)

    print(f"Answer: {full_response}\n")

    # 결과 저장
    results.append({
        "Source": row['Source'],
        "Source_path": row['Source_path'],
        "Question": question,
        "Answer": full_response
    })

Answering Questions:   0%|          | 0/98 [00:00<?, ?it/s]/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Question: 2022년 혁신창업사업화자금(융자)의 예산은 얼마인가요?


Answering Questions:   1%|          | 1/98 [00:07<11:43,  7.25s/it]

Answer: 2022년 혁신창업사업화자금(융자)의 예산은 2,300,000만원입니다. 


Question: 중소벤처기업부의 혁신창업사업화자금(융자) 사업목적은 무엇인가요?


/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Answering Questions:   2%|▏         | 2/98 [00:22<18:51, 11.79s/it]

...

Answer: 국가결산보고서는 국가회계기준에 따라 우발부채를 재무제표상 부채로 인식하지 않고, 주석에 공시합니다. 반면, 지방자치단체 회계기준은 우발부채 발생 가능성과 금액의 추정 가능성에 따라 재무제표상 재정상태표에 인식하는 경우도 있다고 합니다. 즉, 국가결산보고서에서는 우발부채를 주석에만 공시하는 반면, 지방자치단체는 재무제표에 포함될 수도 있다는 점에서 차이가 있습니다. 




Question: 우발부채란 무엇이며, 그 관리가 왜 중요한가?


/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Answering Questions:  99%|█████████▉| 97/98 [26:50<00:20, 20.25s/it]

Answer: 우발부채는 미래에 특정 사건이 발생하지 않는 한 발생하지 않는 의무입니다. 즉, 현재의무로 인식되지 않지만, 특정 조건이 충족되면 금융거래로 인식될 수 있는 잠재적 의무 또는 금액을 신뢰성 있게 추정하기 어려운 현재의무를 의미합니다.

    공공부문에서는 재정건전성과 재정위기 관리 측면에서 우발부채 관리가 매우 중요합니다. 국제기준 재정통계 산출에 발생주의 회계기준이 적용되면서 미래의 다양한 의무를 포괄하는 우발부채 관리의 중요성이 더욱 부각되고 있습니다. 

    우발부채는 재정위험을 발생시키는 구체적인 원인이 될 수 있으며, 예상치 못한 사건으로 인해 발생할 수 있습니다. 따라서 우발부채를 적절하게 파악하고 관리함으로써 재정적 안정성을 확보하고 예상치 못한 재정적 어려움을 예방할 수 있습니다. 




Question: 보증이란 무엇이며, 어떤 형태의 보증이 재정상태표에 부채로 기록되는가? 또한 표준화 보증이란 무엇이며, 어떤 목적으로 발행되는가?


/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Answering Questions: 100%|██████████| 98/98 [27:20<00:00, 16.74s/it]

Answer: 보증은 특정 의무를 이행하도록 다른 당사자를 보장하는 약속입니다. 

    모든 보증이 재정상태표에 부채로 기록되는 것은 아닙니다. 파생금융상품 형태의 보증이나 표준화된 보증 제도 하에서 설정하는 충당부채 형태의 보증은 재정상태표에 부채로 기록됩니다.

    표준화 보증은 통상 아주 적은 금액에 대해 획일적 조건으로 대규모로 발행하는 보증입니다. 수출(무역)신용 보증, 환보증, 다양한 종류의 보험(예금, 농작물, 자연재해 등), 농민융자, 모기지론, 학자금융자, 중소기업융자 등이 있습니다. 주로 유사한 특징을 지니고 위험 분산을 목적으로 하는 보증들이 많으며, 보증인은 확률통계에 기반하여 평균손실(채무불이행률)을 추정할 수 있습니다. 






In [ ]:
# 제출용 샘플 파일 로드
submit_df = pd.read_csv("./sample_submission.csv")

# 생성된 답변을 제출 DataFrame에 추가
submit_df['Answer'] = [item['Answer'] for item in results]
submit_df['Answer'] = submit_df['Answer'].fillna("데이콘")     # 모델에서 빈 값 (NaN) 생성 시 채점에 오류가 날 수 있음 [ 주의 ]

# 결과를 CSV 파일로 저장
submit_df.to_csv("./baseline_submission.csv", encoding='UTF-8-sig', index=False)

: 